# Naive Bayes: update a belief with evidence

Companion to HTB Academy Module 290, Section 7. First we reproduce the page's medical-test arithmetic. Then a **synthetic** two-feature spam example shows the Naive Bayes product. This is a teaching calculation, not a diagnostic tool or a trained security filter.

Read $P(A\mid B)$ as *the probability of A given B*. Read $P(A\mid B)=P(B\mid A)P(A)/P(B)$ as *the probability of A given B equals the probability of B given A times the prior probability of A, divided by the overall probability of B*. The direction across the word **given** matters.

In [ ]:
# HTB's inputs: A = has the disease; B = tests positive.
p_disease = 0.01             # P(A), the prior prevalence.
p_positive_if_disease = 0.95  # P(B|A), sensitivity.
p_positive_if_healthy = 0.05  # P(B|not A), the false-positive rate.
p_healthy = 1 - p_disease    # P(not A); the two groups cover everyone.

# Law of total probability: P(B) = P(B|A)P(A) + P(B|not A)P(not A).
p_positive = (p_positive_if_disease * p_disease
              + p_positive_if_healthy * p_healthy)
# Bayes' theorem: P(A|B) = P(B|A)P(A) / P(B).
p_disease_if_positive = p_positive_if_disease * p_disease / p_positive
print(f'Overall positive-test chance: {p_positive:.3f}')
print(f'Disease chance after a positive test: {p_disease_if_positive:.1%}')
assert abs(p_positive - 0.059) < 1e-12
assert abs(p_disease_if_positive - 0.0095 / 0.059) < 1e-12

## Turn the same probabilities into expected counts

For 10,000 hypothetical people, the positive group includes both true and false positives. The fraction of positive tests that are true positives must match the posterior above. These are **expected counts** under HTB's assumptions; they do not describe actual patients.

In [ ]:
population = 10_000
true_positives = population * p_disease * p_positive_if_disease
false_positives = population * p_healthy * p_positive_if_healthy
positive_tests = true_positives + false_positives
print(f'True positives: {true_positives:.0f}; false positives: {false_positives:.0f}')
print(f'True positives / all positives: {true_positives:.0f}/{positive_tests:.0f}')
assert (true_positives, false_positives, positive_tests) == (95, 495, 590)
assert abs(true_positives / positive_tests - p_disease_if_positive) < 1e-12

## A small Bernoulli Naive Bayes classification

Read $P(C)\prod_iP(x_i\mid C)$ as *the prior for class C times the product over features i of each feature's likelihood given C*. Capital pi ($\prod$) means **multiply the terms**. Here each feature is binary: does a word appear? The message has `free` but lacks `meeting`, so the latter contributes a probability of **absence**, $1-P(\text{meeting present}\mid C)$. We score both classes, then divide by their score total so the resulting posteriors sum to one. HTB explains this approach; the particular word probabilities below are invented for clarity.

In [ ]:
# Class priors: before reading the message, 20% are spam in this toy world.
prior_spam, prior_ham = 0.2, 0.8
# Conditional word-presence probabilities, supplied as illustrative inputs.
p_free_if_spam, p_meeting_if_spam = 0.7, 0.1
p_free_if_ham, p_meeting_if_ham = 0.1, 0.6

# Message features: free is present; meeting is absent.
# Naive assumption: multiply the two feature likelihoods once class is fixed.
spam_score = prior_spam * p_free_if_spam * (1 - p_meeting_if_spam)
ham_score = prior_ham * p_free_if_ham * (1 - p_meeting_if_ham)
# Normalize the two unnormalized scores to obtain class posteriors.
score_total = spam_score + ham_score
p_spam_given_words = spam_score / score_total
p_ham_given_words = ham_score / score_total
prediction = 'spam' if p_spam_given_words > p_ham_given_words else 'not spam'
print(f'Scores — spam: {spam_score:.3f}; not spam: {ham_score:.3f}')
print(f'Posterior spam chance: {p_spam_given_words:.1%}; prediction: {prediction}')
assert abs(spam_score - 0.126) < 1e-12
assert abs(ham_score - 0.032) < 1e-12
assert abs(p_spam_given_words + p_ham_given_words - 1) < 1e-12
assert prediction == 'spam'

## Boundaries of this example

These probabilities were **not learned from labeled messages**, and words can be correlated even after the class is known. The printed 79.7% is a result *inside this toy model*, not a calibrated probability for a real inbox. HTB also names Gaussian Naive Bayes for continuous measurements and Multinomial Naive Bayes for discrete counts; this notebook uses only the binary-feature Bernoulli variant to match the tiny present/absent example.